In [0]:
%run "/Workspace/Users/kevinraj.arul@diggibyte.com/Databricks_assignment/src/Quetion 1/source_to_bronze/Utils"

In [0]:
%python
from pyspark.sql import functions as F
spark.sql("Create database if not exists assignment.gold")

# Step 1️⃣ - Read the Delta table from Silver layer
silver_table = "assignment.Employee_info.dim_employee"
emp_df = spark.table(silver_table)

# Step 2️⃣ - Add at_load_date column
emp_df = emp_df.withColumn("at_load_date", F.current_date())

# Step 3️⃣ - Salary of each department (descending order)
salary_df = (
    emp_df.groupBy("department_id")
    .agg(F.sum("salary").alias("total_salary"))
    .orderBy(F.desc("total_salary"))
    .withColumn("at_load_date", F.current_date())
)

# Step 4️⃣ - Number of employees per department per country
emp_count_df = (
    emp_df.groupBy("department_id", "country_id")
    .agg(F.count("*").alias("employee_count"))
    .withColumn("at_load_date", F.current_date())
)

# Step 5️⃣ - Department and country IDs (distinct)
dept_country_df = (
    emp_df.select("department_id", "country_id")
    .distinct()
    .withColumn("at_load_date", F.current_date())
)

# Step 6️⃣ - Average age per department
avg_age_df = (
    emp_df.groupBy("department_id")
    .agg(F.round(F.avg("age"), 2).alias("avg_age"))
    .withColumn("at_load_date", F.current_date())
)

# Step 7️⃣ - Write results to Gold layer (Delta format)
salary_df.write.format("delta").mode("overwrite").saveAsTable("assignment.gold.employee_fact")
emp_count_df.write.format("delta").mode("overwrite").saveAsTable("assignment.gold.employee_count_fact")
dept_country_df.write.format("delta").mode("overwrite").saveAsTable("assignment.gold.fact_dept_country")
avg_age_df.write.format("delta").mode("overwrite").saveAsTable("assignment.gold.fact_avg_age")

print("✅ Gold layer created successfully and data written to:")
print("/Volumes/assignment/default/gold/employee/")

sql


In [0]:
%sql
select * from assignment.gold.employee_count_fact

In [0]:
%sql
select * from assignment.gold.employee_fact

In [0]:
%sql
select * from assignment.gold.fact_dept_country

In [0]:
%sql
select * from assignment.gold.fact_avg_age